Note: there are important features that are not compatible, see:
- https://platform.claude.com/docs/en/build-with-claude/extended-thinking#feature-compatibility


## Extended thinking
Gives the model time to work through complex problems before generating a final response. 

![alt text](images/features_extended_thinking.jpg)

## Key benefits
- Better reasoning capabilities for complex tasks
- Increased accuracy on difficult problems
- Transparency into Claude's thought process

# Tradeoffs
- Higher costs (you pay for thinking tokens)
- Increased latency ( thinking takes time )
- More complex response handling in your code

## When to use extended thinking?
The decision is straightforward: use your prompt evaluations. Run your prompts without thinking first, and if the accuracy isn't meeting your requirements after you've already optimized your prompt, then consider enabling extended thinking. It's a tool for when standard prompting isn't quite getting you there

## Response structure and security
Extended thibking responses include a special signature system for security

![alt text](images/features_extended_thinking_security.jpg)

The signature is a cryptographic token that ensures you haven't modified the thinking text. This prevents developers from tampering with Claude's reasoning process, which could potentially lead the model in unsafe directions

## Redacted thinking
Sometimes you'll reviece a redacted thinking block instead of readable reasoning text, this happens when claude's thinking process gets flagged by internal safety systems. The redacted content contains the actual thinking in encrypted form, allowing you to pass the complete message back to Claude in the future conversations without losing context

In [ ]:
## In code:
def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=True,
    thinking_budget=1024
): ...

## Testing redacted responses
For testing purposes, you can force Claude to return a redacted thinking block by sending a special trigger string. This helps ensure your application handles redacted responses gracefully without crashing.

Extended thinking is a powerful feature when you need Claude to tackle complex reasoning tasks, but use it judiciously given the cost and latency implications. Start with standard prompting, optimize thoroughly, then add thinking when you need that extra reasoning capability

## Claude image support
Vision capabilities let you include images in your messages and ask Calude to analyze them in countless ways. You can ask claude to describe what's in an image, compare multiple images, count objects, or perform complex visual analysis tasks

## Image handling basics
![alt text](images/features_images.jpg)

- Up to 100 images across all messages in a single request
- Max size of 5MB per image
- When sending one image: max height/width of 8000px
- When sending multiple images: max height/width of 2000 px
- Images can be included as bas64 encoding or a URL to the image
- Each image counts as tokens based on its dimensions. 
Tokens = (width px * height px) / 750

In [ ]:
with open("image.png", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

add_user_message(messages, [
    # Image Block
    {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": image_bytes,
        }
    },
    # Text Block
    {
        "type": "text",
        "text": "What do you see in this image?"
    }
])

Message flow
![alt text](images/features_images_flow.jpg)

## Prompting tecniques in images

- All the same prompting engineering techniques apply to images
- You can increase claude's vision accuracy by providing guidelines, analysis steps, or by using one-shot/multi-shot examples

## Step by step analysis
![alt text](images/features_images_prompting_techniques_step_by_step.jpg)

Instead of a simple question, provide Claude with a methodology

e.g.

Analyze this image of marbles and determine the exact count using this methodology:

1. Begin by identifying each unique marble one at the time. Assign each number as you identify ...
2. Verify your result by counting with a different method. Start from the bottom-left ...

What is the exact, verified number of marbles in this image?

## One shot examples
You can also improve accuracy by providing examples within you message. Include an image with a known count, state the correct answer, then ask about your target image. This gives Claude a reference point for the type of analysis you want


![alt text](images/features_images_prompting_techniques_one_shot.jpg)

## Real world example
Fire risk assessments
- In some parts of the US, home fire insurance companies requires homeowners to trim or remove trees around the residence
- Sending an inspector out to each property would be expensive
- Solution: get high resolution, up-to-date satellite imagery, ask Claude for a fire risk assessment

The system analyzes satellite images to identify:
- Dense, close-packed trees near the residence
- Difficult access routes for emergency services
- Branches overhanging the residence

Prompt example:
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75%+)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure

4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the home
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple vulnerability points
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numerical rating.

## Extracting pdfs



In [ ]:
with open("earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(
    messages,
    [
        {
            "type": "document",
            "source": {
                "type": "base64",
                "media_type": "application/pdf",
                "data": file_bytes,
            },
        },
        {"type": "text", "text": "Summarize the document in one sentence"},
    ],
)

chat(messages)

Key Changes from Image Processing
When adapting your image processing code for PDFs, you need to update several elements:

- Change the file extension from .png to .pdf
- Update the variable name from image_bytes to file_bytes for clarity
Set the type to "document" instead of "image"
- Change the media type to "application/pdf" instead of "image/png"


## What claude can extract from PDFs

- Text content throughout the document
- Images and charts embedded in the PDF
- Tables and their data relationships
- Document structure and formatting

## Enable citations over a PDF
Add two new fields

In [ ]:
{
    "type": "document",
    "source": {
        "type": "base64",
        "media_type": "application/pdf",
        "data": file_bytes,
    },
    "title": "earth.pdf",
    "citations": { "enabled": True }
}


![alt text](images/features_documents_citated_documents.jpg)

## Citations with Plain Text
Citations aren't limited to PDF documents. You can also use them with plain text sources. When working with text, modify your document structure like this:



In [ ]:
{
    "type": "document", 
    "source": {
        "type": "text",
        "media_type": "text/plain",
        "data": article_text,
    },
    "title": "earth_article",
    "citations": { "enabled": True }
}

## Use cases

- Users need to verify information for accuracy
- You're working with authoritative documents that users should be able to reference
- Transparency about information sources is critical for your application
- Users might want to explore the broader context around specific facts

## Prompt caching
![alt text](images/features_promp_catching.jpg)

## Key idea
When user ask to claude, claude needs to:
- tokenize the prompt
- create embeddings for each token
- add context based on sorrounding text
- generate output text

Every single turn, claude usually deletes that previous tokenization and embedding creation to trash, prompt catching is storing that into caché

## Advantages of prompt caching

- Faster responses: Requests using cached content execute more quickly
- Lower costs: You pay less for the cached protions of your requests
- Automatic optimization: The initial request writes to the cache, follow-up requests read from it

However, there are important limitations to keep in mind:

- cache duration: cached content only lives for one hour
- limited use cases: only beneficial when you're repreatedly sending the same content
- high frequency requirement: Most effective when the same content appears extremely frequently in your requests

Prompt caching works best for scenarios like document analysis workflows, where you're asking multiple questions about the same large document, or iterative editing tasks where the base content remains constant while you refine specific aspects

## Rules of prompt caching
Cache breakpoints
- Work done on messages is not cached automatically
- We have to automatically add 'cache breakpoint' to a block
- Work done for everything before the breakpoint will be cached
- Cache will only be used on follow up requests if the content up to and including the breakpoint is identical

![alt text](images/features_prompt_cache_breakpoint.jpg)

## Shorthand vs Longhand text block

We've been using shorthand for writing a text block, but for cache control we need to use longhand for writing a textblock

![alt text](images/features_prompt_cache_short_hand_vs_longhand_text_block.jpg)

Remember, in followup requests, content must be identical to cache

Note: Cache breakpoints can span across multiple messages and message types. If ypu place a breakpoint in a later message, all previous messages (user, assistant, etc.) will be included in the cached content

![alt text](images/features_prompt_cache_span_across.jpg)

We are not limited to text blocks - cache breakpoints can be added to:
- System prompts
- Tools definitions
- Image blocks
- Tool use and tool result block

![alt text](images/features_prompt_cache_breakpoint_location.jpg)


## VERY IMPORTANT NOTE
System prompts and tool definitions are excellent candidates for caching since they rarely change between requests. This is often where you'll get the most benefit from prompt caching

## Cache ordering
Behind the scenes, Claude processess your request components in a specific order:
- Tools first
- System prompt
- Messages

Understanding this orders helps you place breakpoints effectively

## Important notes about cache
- We can have up to 4 cache breakpoints
- Minimum content lenght, content must be at least 1024 tokens long to be cached (sum of all messages/blocks you're trying to cache)

## Files API

Similar to images
![alt text](images/features_compare_with_images.png)

With the file api we can make individual request ahead of time to upload a particular file.

We get back a file metadata, the most important is 

## File ID

Later on, a user can submit a message where he points to the uploaded file id

![alt text](images/features_files_api_upload_request.png)

## Code execution tool
Server based tool, we do not provide implementation, we provide predefined schema.

Behind the scenes claude decides if runs it in dokcer container

Claude can run code in this containers, whatever code prints sends to cluade, interpret results, send message. These Docker containers doesnt have outside APIs!

## Example of both running
![alt text](images/features_file_api_and_code_execution_example.png)